# OpenAlex AI journal dataset — v1.0

This notebook downloads one raw OpenAlex dataset and creates one lightly cleaned table for the project.

## Scope and outputs

The dataset contains non-retracted journal articles published from 2015 to 2024 whose primary OpenAlex topic is in the agreed AI list and whose journal is in the frozen 64-journal population.

The notebook creates only:

- `openalex_ai_raw_v1_0.jsonl`: complete OpenAlex work records, unchanged and one per line
- `openalex_ai_semiclean_v1_0.csv`: one row per article with the main fields and compact JSON columns for authorships, topics, and keywords

APC information is retained when OpenAlex provides it, but it is not an eligibility requirement. This dataset represents publications, not submissions or author motives.

In [1]:
from pathlib import Path
import getpass
import json
import os
import time
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd

DATASET_VERSION = "1.0"
FREEZE_DATE = "2026-08-01"
START_YEAR = 2015
END_YEAR = 2024

DATA_DIR = Path("openalex_ai_dataset_v1_0")
RAW_PATH = DATA_DIR / "openalex_ai_raw_v1_0.jsonl"
CLEAN_PATH = DATA_DIR / "openalex_ai_semiclean_v1_0.csv"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset version: {DATASET_VERSION}")
print(f"Output folder: {DATA_DIR.resolve()}")

Dataset version: 1.0
Output folder: C:\Users\lenna\OneDrive\Dokumente\Applied AI\Q4\openalex_ai_dataset_v1_0


## 1. Frozen selection

Version 1.0 freezes the 64 journals and 49 included primary topics established in the earlier eligibility review. The journal population had current DOAJ and fully-OA status, at least 100 matching articles during 2015–2024, and activity in at least five years; APC values did not control inclusion.

The freeze fixes the selection rules and IDs. The raw JSONL becomes the fixed data file for version 1.0 when this notebook is first run.

In [2]:
# These IDs are the complete frozen population for notebook version 1.0.
FROZEN_JOURNAL_IDS = '''
S101949793 S107516304 S125501549 S127898559 S137468011 S139930977 S149016011 S155526855
S167961193 S17147534 S178776955 S19032547 S190629608 S190680769 S195231649 S196734849
S198098182 S20211220 S202381698 S207319443 S2485537415 S24978797 S2595095599 S2596394214
S2729999759 S2737955091 S2738286576 S2764413287 S2764650051 S2764846071 S2764955546 S2765015152
S2898612692 S3035462843 S30889260 S32837994 S34838331 S3569471 S4210172076 S4210173132
S4210177785 S4210178049 S4210192031 S4210195431 S4210196574 S4210197006 S4210200687 S4210201048
S4210205812 S4210206423 S4210213891 S4210214273 S4210216316 S4210219776 S4210228075 S4210228265
S4210238752 S43295729 S4511983 S52395412 S64187185 S83215360 S9692511 S997959834
'''.split()

INCLUDED_TOPIC_IDS = '''
T10028 T10100 T10181 T10201 T10215 T10320 T10456 T10462 T10637 T10664
T10711 T10820 T10862 T10906 T11010 T11273 T11303 T11307 T11512 T11550
T11574 T11612 T11652 T11689 T11901 T11902 T11975 T12026 T12031 T12072
T12128 T12131 T12262 T12380 T12535 T12611 T12676 T12761 T12814 T13062
T13083 T13567 T13629 T13674 T13702 T13851 T13904 T14175 T14335
'''.split()

assert len(FROZEN_JOURNAL_IDS) == len(set(FROZEN_JOURNAL_IDS)) == 64
assert len(INCLUDED_TOPIC_IDS) == len(set(INCLUDED_TOPIC_IDS)) == 49

print(f"Frozen journals: {len(FROZEN_JOURNAL_IDS)}")
print(f"Included primary topics: {len(INCLUDED_TOPIC_IDS)}")

Frozen journals: 64
Included primary topics: 49


## 2. Download the raw records

OpenAlex is queried with cursor pagination and 100 records per request. Each returned work object is written unchanged to the JSONL file; pagination wrappers are not stored.

A free OpenAlex API key is required. The key is requested securely and is never written to the notebook or output files.

In [3]:
API_URL = "https://api.openalex.org/works"


def build_work_filter():
    parts = [
        f"from_publication_date:{START_YEAR}-01-01",
        f"to_publication_date:{END_YEAR}-12-31",
        "type:article",
        "is_retracted:false",
        "primary_location.source.type:journal",
        f"primary_location.source.id:{'|'.join(FROZEN_JOURNAL_IDS)}",
        f"primary_topic.id:{'|'.join(INCLUDED_TOPIC_IDS)}",
    ]
    return ",".join(parts)


def fetch_page(params, attempts=6):
    url = f"{API_URL}?{urlencode(params)}"
    last_error = None

    for attempt in range(attempts):
        request = Request(
            url,
            headers={"User-Agent": "openalex-ai-dataset-notebook/1.0"},
        )
        try:
            with urlopen(request, timeout=90) as response:
                return json.loads(response.read().decode("utf-8"))
        except HTTPError as error:
            last_error = error
            if error.code not in {429, 500, 502, 503, 504}:
                detail = error.read().decode("utf-8", errors="replace")[:1000]
                raise RuntimeError(f"OpenAlex HTTP {error.code}: {detail}") from error
            retry_after = error.headers.get("Retry-After")
            wait_seconds = float(retry_after) if retry_after else min(30, 2 ** attempt)
        except (URLError, TimeoutError) as error:
            last_error = error
            wait_seconds = min(30, 2 ** attempt)

        if attempt < attempts - 1:
            time.sleep(wait_seconds)

    raise RuntimeError("OpenAlex request failed after repeated attempts.") from last_error


def download_raw(api_key, destination):
    temporary_path = destination.with_suffix(".tmp.jsonl")
    cursor = "*"
    page = 0
    record_count = 0
    reported_count = None
    cost_usd = 0.0

    with temporary_path.open("w", encoding="utf-8", newline="\n") as handle:
        while cursor:
            payload = fetch_page(
                {
                    "api_key": api_key,
                    "filter": build_work_filter(),
                    "per_page": 100,
                    "cursor": cursor,
                }
            )
            results = payload.get("results", [])
            meta = payload.get("meta", {})

            if reported_count is None:
                reported_count = int(meta.get("count", len(results)))

            for work in results:
                handle.write(json.dumps(work, ensure_ascii=False, separators=(",", ":")) + "\n")

            page += 1
            record_count += len(results)
            cost_usd += float(meta.get("cost_usd", 0.0) or 0.0)
            cursor = meta.get("next_cursor")

            if page == 1 or page % 25 == 0 or not cursor:
                print(f"Page {page}: {record_count:,} records")

    if record_count == 0:
        temporary_path.unlink(missing_ok=True)
        raise RuntimeError("OpenAlex returned no records.")
    if reported_count is not None and record_count != reported_count:
        temporary_path.unlink(missing_ok=True)
        raise RuntimeError(
            f"Downloaded {record_count:,} records, but OpenAlex reported {reported_count:,}. "
            "Run the download again so the raw file is complete."
        )

    temporary_path.replace(destination)
    return {"records": record_count, "pages": page, "cost_usd": cost_usd}

In [4]:
if RAW_PATH.exists():
    print(f"Using the existing frozen raw file: {RAW_PATH}")
else:
    api_key = os.environ.get("OPENALEX_API_KEY", "").strip()
    if not api_key:
        api_key = getpass.getpass("OpenAlex API key: ").strip()
    if not api_key:
        raise ValueError("An OpenAlex API key is required for the download.")

    download_summary = download_raw(api_key, RAW_PATH)
    print(
        f"Saved {download_summary['records']:,} records from "
        f"{download_summary['pages']} pages to {RAW_PATH}."
    )
    print(f"Reported API cost: ${download_summary['cost_usd']:.4f}")

Page 1: 100 records
Page 25: 2,500 records
Page 50: 5,000 records
Page 75: 7,500 records
Page 100: 10,000 records
Page 125: 12,500 records
Page 150: 15,000 records
Page 175: 17,500 records
Page 200: 20,000 records
Page 225: 22,500 records
Page 250: 25,000 records
Page 275: 27,400 records
Saved 27,400 records from 275 pages to openalex_ai_dataset_v1_0\openalex_ai_raw_v1_0.jsonl.
Reported API cost: $0.0275


## 3. Create the semi-clean table

The raw records remain unchanged. The semi-clean file reconstructs abstracts, uses short OpenAlex IDs, and keeps nested authorships, topics, and keywords as JSON text so that each article stays on one row.

Raw affiliation strings are omitted from the semi-clean file because structured institution data is sufficient for the planned analysis.

In [5]:
def short_id(value):
    if value is None or value == "":
        return None
    return str(value).rstrip("/").split("/")[-1]


def clean_doi(value):
    if not value:
        return None
    return str(value).replace("https://doi.org/", "", 1)


def abstract_from_index(index):
    if not index:
        return None
    positions = [position for values in index.values() for position in values]
    if not positions:
        return None
    words = [None] * (max(positions) + 1)
    for word, values in index.items():
        for position in values:
            words[position] = word
    return " ".join(word for word in words if word is not None)


def json_text(value):
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))


def usd_value(value):
    if not isinstance(value, dict):
        return None
    return value.get("value_usd")

In [6]:
def compact_authorships(work):
    authorships = []
    for item in work.get("authorships") or []:
        author = item.get("author") or {}
        institutions = []
        for institution in item.get("institutions") or []:
            institutions.append(
                {
                    "institution_id": short_id(institution.get("id")),
                    "institution_name": institution.get("display_name"),
                    "ror": institution.get("ror"),
                    "country_code": institution.get("country_code"),
                    "type": institution.get("type"),
                }
            )
        authorships.append(
            {
                "author_id": short_id(author.get("id")),
                "author_name": author.get("display_name") or item.get("raw_author_name"),
                "orcid": short_id(author.get("orcid")),
                "position": item.get("author_position"),
                "is_corresponding": bool(item.get("is_corresponding")),
                "countries": item.get("countries") or [],
                "institutions": institutions,
            }
        )
    return authorships


def compact_topics(work):
    return [
        {
            "topic_id": short_id(topic.get("id")),
            "topic_name": topic.get("display_name"),
            "score": topic.get("score"),
            "subfield_id": short_id((topic.get("subfield") or {}).get("id")),
            "subfield_name": (topic.get("subfield") or {}).get("display_name"),
        }
        for topic in (work.get("topics") or [])
    ]


def compact_keywords(work):
    return [
        {
            "keyword_id": short_id(keyword.get("id")),
            "keyword_name": keyword.get("display_name"),
            "score": keyword.get("score"),
        }
        for keyword in (work.get("keywords") or [])
    ]

In [7]:
def work_to_row(work):
    location = work.get("primary_location") or {}
    source = location.get("source") or {}
    primary_topic = work.get("primary_topic") or {}
    subfield = primary_topic.get("subfield") or {}
    open_access = work.get("open_access") or {}
    best_oa_location = work.get("best_oa_location") or {}
    authorships = compact_authorships(work)
    topics = compact_topics(work)
    keywords = compact_keywords(work)

    return {
        "work_id": short_id(work.get("id")),
        "doi": clean_doi(work.get("doi")),
        "title": work.get("title") or work.get("display_name"),
        "abstract": abstract_from_index(work.get("abstract_inverted_index")),
        "publication_year": work.get("publication_year"),
        "publication_date": work.get("publication_date"),
        "work_type": work.get("type"),
        "language": work.get("language"),
        "is_retracted": work.get("is_retracted"),
        "cited_by_count": work.get("cited_by_count"),
        "fwci": work.get("fwci"),
        "referenced_works_count": work.get("referenced_works_count"),
        "journal_id": short_id(source.get("id")),
        "journal_name": source.get("display_name"),
        "journal_issn_l": source.get("issn_l"),
        "journal_is_oa_current": source.get("is_oa"),
        "journal_is_in_doaj_current": source.get("is_in_doaj"),
        "publisher_id_current": short_id(source.get("host_organization")),
        "publisher_name_current": source.get("host_organization_name"),
        "article_is_oa": open_access.get("is_oa"),
        "article_oa_status": open_access.get("oa_status"),
        "article_oa_url": open_access.get("oa_url"),
        "article_license": best_oa_location.get("license") or location.get("license"),
        "apc_list_usd": usd_value(work.get("apc_list")),
        "apc_paid_usd": usd_value(work.get("apc_paid")),
        "primary_topic_id": short_id(primary_topic.get("id")),
        "primary_topic_name": primary_topic.get("display_name"),
        "primary_topic_score": primary_topic.get("score"),
        "primary_subfield_id": short_id(subfield.get("id")),
        "primary_subfield_name": subfield.get("display_name"),
        "author_count": len(authorships),
        "corresponding_author_count": sum(
            bool(item["is_corresponding"]) for item in authorships
        ),
        "authorships_json": json_text(authorships),
        "topics_json": json_text(topics),
        "keywords_json": json_text(keywords),
        "external_ids_json": json_text(work.get("ids") or {}),
        "indexed_in_json": json_text(work.get("indexed_in") or []),
        "openalex_created_date": work.get("created_date"),
        "openalex_updated_date": work.get("updated_date"),
    }

In [8]:
if not RAW_PATH.exists():
    raise FileNotFoundError("Run the download section before creating the semi-clean table.")

rows = []
raw_record_count = 0
with RAW_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue
        try:
            work = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(f"Invalid JSON on raw line {line_number}.") from error
        rows.append(work_to_row(work))
        raw_record_count += 1

works = pd.DataFrame(rows)
print(f"Parsed {raw_record_count:,} raw records.")
works.head(3)

Parsed 27,400 raw records.


,work_id,doi,title,abstract,publication_year,publication_date,work_type,language,is_retracted,cited_by_count,...,primary_subfield_name,author_count,corresponding_author_count,authorships_json,topics_json,keywords_json,external_ids_json,indexed_in_json,openalex_created_date,openalex_updated_date
0,W2493916176,10.1162/tacl_a_00051,Enriching Word Vectors with Subword Information,"Continuous word representations, trained on la...",2017,2017-12-01,article,en,False,9830,...,Artificial Intelligence,4,4,"[{""author_id"":""A5035420035"",""author_name"":""Pio...","[{""topic_id"":""T10028"",""topic_name"":""Topic Mode...","[{""keyword_id"":""computer-science"",""keyword_nam...","{""openalex"":""https://openalex.org/W2493916176""...","[""crossref"",""doaj""]",2025-10-10T00:00:00,2026-07-31T08:31:51.225901
1,W2395579298,10.1186/s40537-016-0043-6,A survey of transfer learning,Machine learning and data mining techniques ha...,2016,2016-05-28,article,en,False,6220,...,Artificial Intelligence,3,1,"[{""author_id"":""A5035627801"",""author_name"":""Kar...","[{""topic_id"":""T11307"",""topic_name"":""Domain Ada...","[{""keyword_id"":""computer-science"",""keyword_nam...","{""openalex"":""https://openalex.org/W2395579298""...","[""crossref"",""doaj""]",2025-10-10T00:00:00,2026-07-31T08:31:51.225901
2,W2891503716,10.1109/access.2018.2870052,Peeking Inside the Black-Box: A Survey on Expl...,At the dawn of the fourth industrial revolutio...,2018,2018-01-01,article,en,False,5964,...,Artificial Intelligence,2,0,"[{""author_id"":""A5002938989"",""author_name"":""Ami...","[{""topic_id"":""T12026"",""topic_name"":""Explainabl...","[{""keyword_id"":""transparency"",""keyword_name"":""...","{""openalex"":""https://openalex.org/W2891503716""...","[""crossref"",""doaj""]",2025-10-10T00:00:00,2026-07-31T08:31:51.225901


## 4. Validate and save

The final checks verify the article scope, frozen IDs, unique work IDs, and the parsed CSV row count before the file is accepted.

In [9]:
observed_journals = set(works["journal_id"].dropna())
observed_topics = set(works["primary_topic_id"].dropna())
doi_duplicates = int(works.loc[works["doi"].notna(), "doi"].duplicated().sum())

checks = {
    "raw records were parsed": len(works) == raw_record_count and raw_record_count > 0,
    "work IDs are present and unique": works["work_id"].notna().all() and works["work_id"].is_unique,
    "all records are articles": works["work_type"].eq("article").all(),
    "no record is retracted": ~works["is_retracted"].fillna(False).astype(bool).any(),
    "publication years are within scope": works["publication_year"].between(START_YEAR, END_YEAR).all(),
    "only frozen journals occur": observed_journals <= set(FROZEN_JOURNAL_IDS),
    "all frozen journals are represented": observed_journals == set(FROZEN_JOURNAL_IDS),
    "only included primary topics occur": observed_topics <= set(INCLUDED_TOPIC_IDS),
}

failed_checks = [name for name, passed in checks.items() if not passed]
if failed_checks:
    raise AssertionError("Validation failed: " + "; ".join(failed_checks))

temporary_csv = CLEAN_PATH.with_suffix(".tmp.csv")
works.to_csv(temporary_csv, index=False, encoding="utf-8-sig")
written_back = pd.read_csv(temporary_csv, low_memory=False)
if len(written_back) != len(works):
    temporary_csv.unlink(missing_ok=True)
    raise RuntimeError("The written CSV row count does not match the in-memory table.")
temporary_csv.replace(CLEAN_PATH)

summary = pd.DataFrame(
    {
        "value": {
            "articles": len(works),
            "journals": works["journal_id"].nunique(),
            "primary_topics": works["primary_topic_id"].nunique(),
            "years": f"{works['publication_year'].min()}–{works['publication_year'].max()}",
            "abstract_coverage_percent": round(works["abstract"].notna().mean() * 100, 2),
            "doi_coverage_percent": round(works["doi"].notna().mean() * 100, 2),
            "duplicate_dois": doi_duplicates,
        }
    }
)

print(f"Saved the semi-clean table to {CLEAN_PATH}.")
print("All blocking checks passed.")
summary

Saved the semi-clean table to openalex_ai_dataset_v1_0\openalex_ai_semiclean_v1_0.csv.
All blocking checks passed.


,value
articles,27400
journals,64
primary_topics,49
years,2015–2024
abstract_coverage_percent,98.4
doi_coverage_percent,99.35
duplicate_dois,0


## Result

`openalex_ai_raw_v1_0.jsonl` is the immutable source for this dataset version. `openalex_ai_semiclean_v1_0.csv` is the practical baseline for later analysis.

Current publisher, OA, citation, FWCI, and APC fields must not be treated as historical values. OpenAlex also limits a work response to its first 100 authorships.

References: [OpenAlex Works](https://developers.openalex.org/api-reference/works), [filter syntax](https://developers.openalex.org/guides/filtering), and [cursor pagination](https://developers.openalex.org/guides/page-through-results).